# Activity Guide (For Students)

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('../datasets/dados_streaming.csv')
df.sample(5)

,id_cliente,idade,mensalidade,meses_contrato,suporte_tecnico,target
399,1399,22.0,16.01,30,Não,Fiel
233,1233,66.0,18.55,46,Não,Cancelou
241,1241,54.0,12.37,11,Não,Fiel
112,1112,24.0,18.95,8,Sim,Fiel
232,1232,69.0,10.46,28,Sim,Fiel



## Phase 1: Data Audit

### Cleaning: How many customers are missing the age? Fill in these values with the median age.

In [2]:
df['idade'].isnull().sum()

np.int64(15)

In [3]:
df['idade'] = df['idade'].fillna(df['idade'].median())

### Cardinality: How many unique values are there in column suporte_tecnico? And in the target?

In [4]:
print(df['suporte_tecnico'].nunique())
df['suporte_tecnico'].unique()

2


array(['Não', 'Sim'], dtype=object)

In [5]:
print(df['target'].nunique())
df['target'].unique()

2


array(['Fiel', 'Cancelou'], dtype=object)

### Frequency: What percentage of customers have "Canceled" the service? (Tip: use value_counts(normalize=True)).

In [6]:
df['target'].value_counts(normalize=True)

target
Fiel        0.71
Cancelou    0.29
Name: proportion, dtype: float64

## Phase 2: Filters and Logic

### Segmentation: Create a new DataFrame called clientes_premium that contains only those who pay a monthly fee greater than 15.00 and have been in the service for more than 12 months.

In [7]:
clientes_premium = df[(df['mensalidade'] > 15.00) & (df['meses_contrato'] > 12)]
clientes_premium.sample(5)

,id_cliente,idade,mensalidade,meses_contrato,suporte_tecnico,target
50,1050,56.0,16.25,24,Sim,Cancelou
278,1278,33.0,18.52,35,Sim,Fiel
258,1258,61.0,18.67,35,Sim,Fiel
263,1263,72.0,16.10,34,Não,Cancelou
253,1253,50.0,17.62,33,Sim,Cancelou


### Removal: Does the id_cliente column help predict user behavior? If not, remove it from the DataFrame.

In [8]:
df = df.drop('id_cliente', axis= 1)

## Phase 3: Model Preparation (Encoding)

### Manual Mapping: Turn the target column into numbers: True -> 0, Canceled -> 1.

In [9]:
target_map = {
    'Fiel' : 0,
    'Cancelou': 1
}

df['target'] = df['target'].map(target_map)
df.sample(5)

,idade,mensalidade,meses_contrato,suporte_tecnico,target
432,59.0,11.92,34,Não,0
149,52.0,19.37,29,Não,1
448,43.0,12.22,24,Sim,0
240,66.0,18.40,21,Sim,1
54,31.0,14.93,24,Não,0


### Binary Mapping: Transform column suporte_tecnico into: No -> 0, Yes -> 1.

In [10]:
df['suporte_tecnico'] = df['suporte_tecnico'].map({'Não':0,'Sim':1})
df.sample(5)

,idade,mensalidade,meses_contrato,suporte_tecnico,target
45,54.0,14.63,34,0,0
39,68.0,10.31,11,1,0
218,44.0,10.35,27,1,0
420,70.0,18.21,13,1,0
381,39.0,13.88,46,1,1


## Phase 4: ML Insights

### Grouping: What is the average meses_contrato for those who canceled vs. those who are loyal?

In [11]:
df.groupby('target')['meses_contrato'].mean()

target
0    24.247887
1    25.434483
Name: meses_contrato, dtype: float64

### Correlation: Is there any strong relationship between the monthly fee and the numerical target?

In [12]:
df.corr()

,idade,mensalidade,meses_contrato,suporte_tecnico,target
idade,1.000000,0.030189,-0.020279,-0.029992,-0.059960
mensalidade,0.030189,1.000000,0.030037,0.019706,-0.080541
meses_contrato,-0.020279,0.030037,1.000000,0.021316,0.039862
suporte_tecnico,-0.029992,0.019706,0.021316,1.000000,0.006693
target,-0.059960,-0.080541,0.039862,0.006693,1.000000


No

## EXTRA
Create a new variable that helps the model understand the value of each customer. Create a column named "Total_amount"
What you should do:
- Multiply the monthly fee by the number of contract_months.
- Create a business rule: If the Total_amount is greater than 500, the customer is considered "VIP" (1), otherwise it is "Normal" (0).

In [13]:
df['Total_amount'] = df['meses_contrato'] * df['mensalidade']
df['Status'] = df['Total_amount'].map(lambda x: 'VIP' if x > 500 else 'Normal')

df.sample(10)

,idade,mensalidade,meses_contrato,suporte_tecnico,target,Total_amount,Status
214,20.0,16.03,23,0,0,368.69,Normal
64,61.0,13.78,20,1,0,275.60,Normal
318,66.0,18.64,16,0,0,298.24,Normal
250,40.0,11.04,24,1,0,264.96,Normal
71,67.0,18.95,1,0,0,18.95,Normal
313,33.0,16.80,45,0,0,756.00,VIP
180,60.0,10.64,17,0,0,180.88,Normal
88,48.0,17.25,14,0,0,241.50,Normal
310,59.0,11.96,15,0,1,179.40,Normal
276,28.0,13.87,25,1,0,346.75,Normal
